<a href="https://colab.research.google.com/github/samiraghafarigousheh-sys/PyBuildingEnergy_AIBteam_AU/blob/main/copy_of_pybuildingenergy_my_house.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
folder_path='/content/drive/MyDrive/pybuildingenergy_myhouse'
os.makedirs(folder_path, exist_ok=True)


In [ ]:
!git clone https://github.com/Sarthak790/pybuildinenergy_AIB.git
%cd pybuildinenergy_AIB

Cloning into 'pybuildinenergy_AIB'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 126 (delta 44), reused 115 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 3.59 MiB | 11.70 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/pybuildinenergy_AIB


In [ ]:
!pip install pybuildingenergy
import numpy as np
import pandas as pd
import os
import pybuildingenergy as pybui
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI
from pybuildingenergy.source.graphs import Graphs_and_report
from pybuildingenergy.source.utils import ISO52016

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 58.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.0/633.0 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
#1Configuration
weather_file= "AUS_VIC_Melbourne.Airport.948660_TMYx.epw"
weather_source= "epw" if weather_file is not None else "EPW Map"

OUTPUT_DIR = folder_path
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
 #  3) CONSTRUCTION U-VALUES & THERMAL CAPACITY

# U-values (W / m²·K) — Australian BCA 2006 minimum-spec
U_EXT_WALL  = 1.00   # brick veneer / precast w/ R1.0 insulation
U_INT_WALL  = 2.50   # concrete block + plasterboard, no insulation
U_INT_SLAB  = 1.80   # 200 mm concrete intermediate floor
U_WINDOW    = 5.40   # aluminium-frame single glazing
G_WINDOW    = 0.65   # SHGC of clear single glazing

# Solar absorptance — DARK RED BRICK (confirmed from photo)
ABS_EXT_WALL = 0.75  # dark red brick (was 0.55 for mid-tone, corrected after seeing photo)
ABS_INT      = 0.0   # interior surfaces

# Areal thermal capacity (J / m²·K)
C_EXT_WALL = 450_000   # heavy concrete external wall
C_INT_WALL = 330_000   # concrete-block partition
C_INT_SLAB = 480_000   # 200 mm concrete slab
C_WINDOW   = 0


In [ ]:
#2GEometry

LEN_NS  = 5.0   # N-S length, m  (apartment depth — perpendicular to Barry St)
LEN_EW  = 4.0   # E-W length, m  (apartment width — along Barry St)
HEIGHT  = 2.7   # ceiling height, m

FLOOR_AREA = LEN_NS * LEN_EW                # 20.0 m²
VOLUME     = FLOOR_AREA * HEIGHT            # 54.0 m³

# Wall gross areas
A_NORTH_GROSS = LEN_EW * HEIGHT             # 10.8 m² (BARRY ST — EXTERIOR + 2 windows)
A_SOUTH_GROSS = LEN_EW * HEIGHT             # 10.8 m² (to corridor)
A_EAST_GROSS  = LEN_NS * HEIGHT             # 13.5 m² (to Apt 306)
A_WEST_GROSS  = LEN_NS * HEIGHT             # 13.5 m² (to Apt 304)

# Two horizontal-slider windows side-by-side on the north wall
WIN_WIDTH_FIXED,    WIN_HEIGHT_FIXED    = 1.8, 1.4   # fixed glazing
WIN_WIDTH_OPERABLE, WIN_HEIGHT_OPERABLE = 1.8, 1.4   # operable (slider)

A_WINDOW_FIXED    = WIN_WIDTH_FIXED * WIN_HEIGHT_FIXED         # 2.52 m²
A_WINDOW_OPERABLE = WIN_WIDTH_OPERABLE * WIN_HEIGHT_OPERABLE   # 2.52 m²
A_WINDOW_TOTAL    = A_WINDOW_FIXED + A_WINDOW_OPERABLE         # 5.04 m²

# Opaque part of the north wall (= gross − both windows)
A_NORTH_OPAQUE = A_NORTH_GROSS - A_WINDOW_TOTAL                # 5.76 m²


# Sanity check: window-to-wall ratio of the north facade ≈ 47%
# 1.8 + 1.8 = 3.6 m of glazing across a 4 m wide wall = 90% width coverage
# Window heights are 1.4 m on a 2.7 m wall = 52% height coverage

In [ ]:
 # 4 BUILDING DICTIONARY
BUI = {
    # --------- 4a) BUILDING METADATA -----------------------
    "building": {
        "name": "Apt_305_50_Barry_St_Carlton",
        "azimuth_relative_to_true_north": 0,
        "latitude":  -37.800,
        "longitude": 144.968,
        "exposed_perimeter": 0,
        "height": HEIGHT,
        "wall_thickness": 0.20,
        "n_floors": 1,
        "building_type_class": "Residential_apartment",
        "adj_zones_present": True,
        "number_adj_zone": 5,                      # above, below, N, S, corridor (E)
        "net_floor_area": FLOOR_AREA,
        "construction_class": "class_iii",
        "construction_year": "2006-today",
        "country": "Australia",
    },

    # --------- 4b) ADJACENT ZONES ----------------------------------------
    "adjacent_zones": [
        {   # Apt 405 — studio above
            "name": "apt_above",
            "orientation_zone": {"azimuth": 0.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Apt 205 — studio below
            "name": "apt_below",
            "orientation_zone": {"azimuth": 0.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Apt 306 — studio to the EAST
            "name": "apt_east",
            "orientation_zone": {"azimuth": 90.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Apt 304 — studio to the WEST
            "name": "apt_west",
            "orientation_zone": {"azimuth": 180.0},
            "area_facade_elements":    np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": VOLUME,
            "building_type_class": "Residential_apartment",
            "a_use": FLOOR_AREA,
        },
        {   # Building corridor — runs N-S along the east, all interior
            "name": "corridor",
            "orientation_zone": {"azimuth": 180.0},
            "area_facade_elements":    np.array([81.0, 5.4, 81.0, 5.4, 60.0, 60.0]),
            "typology_elements":       ["OP", "OP", "OP", "OP", "OP", "OP"],
            "transmittance_U_elements":np.array([U_INT_WALL]) * 6,
            "orientation_elements":    np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
            "volume": 162.0,
            "building_type_class": "Residential_apartment",
            "a_use": 60.0,
        },
    ],

    # --------- 4c) ENVELOPE SURFACES (west-window layout) -----------
    "building_surface": [

        # 1) NORTH EXTERIOR WALL (opaque part — windows separate below)
        {
            "name": "North exterior wall (opaque)",
            "type": "opaque",
            "area": A_NORTH_OPAQUE,
            "sky_view_factor": 0.5,
            "u_value": U_EXT_WALL,
            "solar_absorptance": ABS_EXT_WALL,
            "thermal_capacity": C_EXT_WALL,
            "orientation": {"azimuth": 0.0, "tilt": 90.0},
            "name_adj_zone": None,
            "height": HEIGHT,
            "length": LEN_EW,
        },

        # 2) SOUTH INTERIOR WALL (to corridor)
        {
            "name": "South wall to corridor",
            "type": "opaque",
            "area": A_SOUTH_GROSS,
            "sky_view_factor": 0.0,
            "u_value": U_INT_WALL,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_WALL,
            "orientation": {"azimuth": 180.0, "tilt": 90.0},
            "name_adj_zone": "corridor",
            "height": HEIGHT,
            "length": LEN_EW,
        },

        # 3) EAST INTERIOR WALL (to Apt 306)
        {
            "name": "East wall to Apt 306",
            "type": "opaque",
            "area": A_EAST_GROSS,
            "sky_view_factor": 0.0,
            "u_value": U_INT_WALL,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_WALL,
            "orientation": {"azimuth": 90.0, "tilt": 90.0},
            "name_adj_zone": "apt_east",
            "height": HEIGHT,
            "length": LEN_NS,
        },

        # 4) WEST INTERIOR WALL (to Apt 304)
        {
            "name": "West wall to Apt 304",
            "type": "opaque",
            "area": A_WEST_GROSS,
            "sky_view_factor": 0.0,
            "u_value": U_INT_WALL,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_WALL,
            "orientation": {"azimuth": 270.0, "tilt": 90.0},
            "name_adj_zone": "apt_west",
            "height": HEIGHT,
            "length": LEN_NS,
        },

        # 5) FLOOR (to Apt 205 below)
        {
            "name": "Floor to Apt 205",
            "type": "opaque",
            "area": FLOOR_AREA,
            "sky_view_factor": 0.0,
            "u_value": U_INT_SLAB,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_SLAB,
            "orientation": {"azimuth": 0.0, "tilt": 0.0},
            "name_adj_zone": "apt_below",
            "height": LEN_NS,
            "length": LEN_EW,
        },

        # 6) CEILING (to Apt 405 above)
        {
            "name": "Ceiling to Apt 405",
            "type": "opaque",
            "area": FLOOR_AREA,
            "sky_view_factor": 0.0,
            "u_value": U_INT_SLAB,
            "solar_absorptance": ABS_INT,
            "thermal_capacity": C_INT_SLAB,
            "orientation": {"azimuth": 0.0, "tilt": 0.0},
            "name_adj_zone": "apt_above",
            "height": LEN_NS,
            "length": LEN_EW,
        },

        # 7) NORTH WINDOW — FIXED glazing
        {
            "name": "North window — fixed",
            "type": "transparent",
            "area": A_WINDOW_FIXED,
            "sky_view_factor": 0.5,
            "u_value": U_WINDOW,
            "solar_absorptance": 0.5,
            "thermal_capacity": C_WINDOW,
            "orientation": {"azimuth": 0.0, "tilt": 90.0},
            "name_adj_zone": None,
            "height": WIN_HEIGHT_FIXED,
            "g_value": G_WINDOW,
            "width": WIN_WIDTH_FIXED,
            "parapet": 1.0,
            "shading": True,
            "shading_type": "horizontal_overhang",
            "width_or_distance_of_shading_elements": 0.05,
            "overhang_proprieties": {
                "width_of_horizontal_overhangs": 0.25,
            },
        },

        # 8) NORTH WINDOW — OPERABLE sash (for ventilation)
        {
            "name": "North window — operable",
            "type": "transparent",
            "area": A_WINDOW_OPERABLE,
            "sky_view_factor": 0.5,
            "u_value": U_WINDOW,
            "solar_absorptance": 0.5,
            "thermal_capacity": C_WINDOW,
            "orientation": {"azimuth": 0.0, "tilt": 90.0},
            "name_adj_zone": None,
            "height": WIN_HEIGHT_OPERABLE,
            "g_value": G_WINDOW,
            "width": WIN_WIDTH_OPERABLE,
            "parapet": 1.0,
            "shading": True,
            "shading_type": "horizontal_overhang",
            "width_or_distance_of_shading_elements": 0.05,
            "overhang_proprieties": {
                "width_of_horizontal_overhangs": 0.25,
            },
        },
    ],

    # --------- 4d) UNITS DOCUMENTATION ---------------------
    "units": {
        "area": "m²",
        "u_value": "W/m²K",
        "thermal_capacity": "J/m²K",
        "azimuth": "degrees (0=N, 90=E, 180=S, 270=W)",
        "tilt": "degrees (0=horizontal, 90=vertical)",
        "internal_gain": "W/m²",
        "HVAC_profile": "0: off, 1: on",
    },

    # --------- 4e) OPERATION -------------------------------
    "building_parameters": {

        "temperature_setpoints": {
            "heating_setpoint": 20.0,
            "heating_setback":  17.0,
            "cooling_setpoint": 25.0,
            "cooling_setback":  28.0,
            "units": "°C",
        },

        "system_capacities": {
            "heating_capacity": 10_000_000.0,
            "cooling_capacity": 10_000_000.0,
            "units": "W",
        },

        "ventilation": {
            "ventilation_type": "occupancy",
            "flow_rate_per_person": 0.3,
            "units": "l/(s m²)",
            "custom_heat_transfer_coefficient_ventilation": None,
            "info": "Per-person flow during occupied hours (EN 16798-1)",
        },

        "internal_gains": [
            {
                "name": "occupants",
                "full_load": 4.0,
                "weekday": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,0.2,0.0,0.0,0.0,0.1,0.1,0.0,0.0,0.3,0.7,1.0,1.0,1.0,1.0,1.0,1.0],
                "weekend": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,0.8,0.8,0.8,0.8,0.5,0.5,0.5,0.8,1.0,1.0,1.0,1.0,1.0,1.0],
            },
            {
                "name": "appliances",
                "full_load": 5.0,
                "weekday": [0.3,0.3,0.3,0.3,0.3,0.3,0.5,0.5,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.4,0.7,0.8,0.9,0.9,0.8,0.6,0.4],
                "weekend": [0.3,0.3,0.3,0.3,0.3,0.3,0.3,0.4,0.6,0.7,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.7,0.8,0.9,0.9,0.8,0.6,0.4],
            },
            {
                "name": "lighting",
                "full_load": 3.0,
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.3,0.3,0.2,0.2,0.2,0.2,0.2,0.2,0.3,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
            },
        ],

        "construction": {
            "wall_thickness": 0.20,
            "thermal_bridges": 1.5,
            "units": "m (thickness), W/mK (thermal bridges)",
        },

        "climate_parameters": {
            "coldest_month": 7,        # JULY in Southern Hemisphere
            "units": "1-12 (January-December)",
        },

        "heating_profile": {
            "weekday": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0],
            "weekend": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0],
        },
        "cooling_profile": {
            "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
            "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
        },
        "ventilation_profile": {
            "weekday": [1.0] * 24,
            "weekend": [1.0] * 24,
        },
    },
}





NameError: name 'U_EXT_WALL' is not defined